# XAI for Sentiment Analysis: TextBlob vs BERT
### Experimental Supplement to: *Explainability in Transformer-Based Sentiment Analysis*
**Author:** Bhavishyasri Matangi | Dr. RVR NRI Deemed to be University

This notebook applies LIME explainability to compare a lexicon-based (TextBlob) and a transformer-based (BERT) sentiment classifier on social media text, demonstrating the XAI gap discussed in the survey.

## Step 1: Install Required Libraries
Run this cell first. It may take 2–3 minutes.

In [ ]:
# Install all required packages
!pip install lime transformers torch textblob matplotlib numpy scikit-learn --quiet

## Step 2: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from textblob import TextBlob
from transformers import pipeline
from lime.lime_text import LimeTextExplainer
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## Step 3: Load Dataset
Social media posts covering sarcasm, slang, mixed sentiment — the challenging cases discussed in the survey.

In [ ]:
# Carefully selected social media posts representing challenging XAI cases
# Includes: sarcasm, negation, slang, mixed sentiment, and direct sentiment

social_media_posts = [
    # Clear positive
    "This product is absolutely amazing, I love it so much!",
    "Had the best day ever, feeling so grateful and happy!",
    
    # Clear negative
    "Worst service ever. Never coming back to this place again.",
    "I'm so frustrated, nothing is working and I've wasted hours.",
    
    # SARCASM — surface positive, actual negative (key XAI challenge)
    "Oh great, another delay. Just what I needed today.",
    "Absolutely brilliant idea to cancel the event last minute!",
    "Wow, waited 3 hours for this? Totally worth it.",
    
    # NEGATION — tricky for models
    "I don't think this is a good idea at all.",
    "Not bad, but not great either. Could be worse.",
    
    # MIXED SENTIMENT
    "The food was great but the service was absolutely terrible.",
    "Love the design, hate the price. Such a disappointment.",
    
    # SOCIAL MEDIA SLANG
    "This is lowkey fire ngl, kinda slept on this brand.",
    "nah this ain't it chief, major L from this company.",
    
    # MENTAL HEALTH / DISTRESS (survey's most sensitive case)
    "I'm fine. Just tired. Everything is fine.",
    "Can't sleep again. Just smiling through it all as usual."
]

labels = [
    "Clear Positive", "Clear Positive",
    "Clear Negative", "Clear Negative",
    "Sarcasm", "Sarcasm", "Sarcasm",
    "Negation", "Negation",
    "Mixed", "Mixed",
    "Slang", "Slang",
    "Distress (Masked)", "Distress (Masked)"
]

print(f"✅ Dataset loaded: {len(social_media_posts)} social media posts")
print("\nPost categories:")
for category in sorted(set(labels)):
    count = labels.count(category)
    print(f"  • {category}: {count} posts")

## Step 4: TextBlob Sentiment Analysis (Lexicon-Based Baseline)

In [ ]:
def textblob_sentiment(text):
    """Returns sentiment label and polarity score using TextBlob."""
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    if polarity > 0.05:
        label = "POSITIVE"
    elif polarity < -0.05:
        label = "NEGATIVE"
    else:
        label = "NEUTRAL"
    return label, round(polarity, 4)

textblob_results = []
print("TextBlob Sentiment Results:")
print("-" * 70)
for i, post in enumerate(social_media_posts):
    label, score = textblob_sentiment(post)
    textblob_results.append((label, score))
    print(f"[{labels[i]:18s}] {label:9s} (score: {score:+.4f})")
    print(f"  → '{post[:65]}...' " if len(post) > 65 else f"  → '{post}'")
    print()

## Step 5: BERT Sentiment Analysis (Transformer-Based)

In [ ]:
# Load pretrained BERT sentiment model (downloads ~500MB, takes a few minutes)
print("Loading BERT model... (this may take 2-3 minutes on first run)")
bert_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    return_all_scores=True
)
print("✅ BERT model loaded!")

bert_results = []
print("\nBERT Sentiment Results:")
print("-" * 70)
for i, post in enumerate(social_media_posts):
    scores = bert_classifier(post)[0]
    # Get the winning label and its score
    best = max(scores, key=lambda x: x['score'])
    label = best['label']
    score = round(best['score'], 4)
    bert_results.append((label, score))
    print(f"[{labels[i]:18s}] {label:9s} (confidence: {score:.4f})")
    print(f"  → '{post[:65]}...' " if len(post) > 65 else f"  → '{post}'")
    print()

## Step 6: Model Disagreement Analysis
Where do the two models disagree? These disagreements reveal the XAI gap.

In [ ]:
print("MODEL DISAGREEMENT ANALYSIS")
print("=" * 75)
print(f"{'Post Category':<20} {'TextBlob':<12} {'BERT':<12} {'Agreement':<12}")
print("-" * 75)

agreements = []
disagreements = []

for i, post in enumerate(social_media_posts):
    tb_label = textblob_results[i][0]
    bert_label = bert_results[i][0]
    
    # Normalize for comparison
    tb_norm = "POSITIVE" if tb_label == "POSITIVE" else "NEGATIVE"
    bert_norm = bert_label  # already POSITIVE/NEGATIVE
    
    if tb_label == "NEUTRAL":
        agree = "⚠️ TB=NEUTRAL"
    elif tb_norm == bert_norm:
        agree = "✅ AGREE"
        agreements.append(i)
    else:
        agree = "❌ DISAGREE"
        disagreements.append(i)
    
    print(f"{labels[i]:<20} {tb_label:<12} {bert_label:<12} {agree}")

print("-" * 75)
print(f"\nTotal disagreements: {len(disagreements)} / {len(social_media_posts)} posts")
print("\n🔍 Key Disagreement Posts (the XAI gap in action):")
for idx in disagreements:
    print(f"  [{labels[idx]}] '{social_media_posts[idx]}'")
    print(f"    TextBlob: {textblob_results[idx][0]} | BERT: {bert_results[idx][0]}")

## Step 7: Visualization 1 — Sentiment Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Sentiment Distribution: TextBlob vs BERT\nOn Challenging Social Media Posts', 
             fontsize=14, fontweight='bold', y=1.02)

colors = {'POSITIVE': '#2ecc71', 'NEGATIVE': '#e74c3c', 'NEUTRAL': '#95a5a6'}

# TextBlob distribution
tb_counts = {'POSITIVE': 0, 'NEGATIVE': 0, 'NEUTRAL': 0}
for label, _ in textblob_results:
    tb_counts[label] += 1

axes[0].bar(tb_counts.keys(), tb_counts.values(), 
            color=[colors[k] for k in tb_counts.keys()], edgecolor='black', linewidth=0.8)
axes[0].set_title('TextBlob (Lexicon-Based)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Posts')
axes[0].set_ylim(0, 10)
for bar, (k, v) in zip(axes[0].patches, tb_counts.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                str(v), ha='center', fontweight='bold')

# BERT distribution
bert_counts = {'POSITIVE': 0, 'NEGATIVE': 0}
for label, _ in bert_results:
    bert_counts[label] = bert_counts.get(label, 0) + 1

axes[1].bar(bert_counts.keys(), bert_counts.values(),
            color=[colors[k] for k in bert_counts.keys()], edgecolor='black', linewidth=0.8)
axes[1].set_title('BERT (Transformer-Based)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Posts')
axes[1].set_ylim(0, 10)
for bar, (k, v) in zip(axes[1].patches, bert_counts.items()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig1_sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 1 saved as 'fig1_sentiment_distribution.png'")

## Step 8: Visualization 2 — Agreement Heatmap by Category

In [ ]:
categories = sorted(set(labels))
category_agreement = {}

for cat in categories:
    indices = [i for i, l in enumerate(labels) if l == cat]
    agree_count = 0
    for idx in indices:
        tb = textblob_results[idx][0]
        bert = bert_results[idx][0]
        tb_norm = "POSITIVE" if tb == "POSITIVE" else "NEGATIVE"
        if tb != "NEUTRAL" and tb_norm == bert:
            agree_count += 1
    category_agreement[cat] = agree_count / len(indices) * 100

fig, ax = plt.subplots(figsize=(12, 5))
cats = list(category_agreement.keys())
vals = list(category_agreement.values())
bar_colors = ['#2ecc71' if v == 100 else '#f39c12' if v >= 50 else '#e74c3c' for v in vals]

bars = ax.barh(cats, vals, color=bar_colors, edgecolor='black', linewidth=0.8, height=0.5)
ax.set_xlabel('Model Agreement Rate (%)', fontsize=11)
ax.set_title('TextBlob vs BERT Agreement Rate by Post Category\n(Lower = Greater XAI Gap)', 
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 110)
ax.axvline(x=50, color='gray', linestyle='--', linewidth=1, label='50% threshold')

for bar, val in zip(bars, vals):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontweight='bold')

green = mpatches.Patch(color='#2ecc71', label='Full Agreement (100%)')
orange = mpatches.Patch(color='#f39c12', label='Partial Agreement (50–99%)')
red = mpatches.Patch(color='#e74c3c', label='Disagreement (<50%)')
ax.legend(handles=[green, orange, red], loc='lower right')

plt.tight_layout()
plt.savefig('fig2_agreement_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 2 saved as 'fig2_agreement_by_category.png'")

## Step 9: LIME Explainability — TextBlob
LIME reveals WHICH words drove TextBlob's prediction.

In [ ]:
# LIME wrapper for TextBlob
def textblob_predict_proba(texts):
    """Returns probability array [P(negative), P(positive)] for LIME."""
    results = []
    for text in texts:
        polarity = TextBlob(text).sentiment.polarity
        # Convert polarity (-1 to 1) to probabilities
        pos_prob = (polarity + 1) / 2
        neg_prob = 1 - pos_prob
        results.append([neg_prob, pos_prob])
    return np.array(results)

explainer = LimeTextExplainer(class_names=['NEGATIVE', 'POSITIVE'])

# Explain the sarcasm example — most interesting case
sarcasm_post = "Oh great, another delay. Just what I needed today."
print(f"Explaining TextBlob on sarcasm post:")
print(f"Post: '{sarcasm_post}'")
print(f"TextBlob prediction: {textblob_sentiment(sarcasm_post)[0]} (score: {textblob_sentiment(sarcasm_post)[1]})")
print("\nGenerating LIME explanation...")

tb_exp = explainer.explain_instance(
    sarcasm_post,
    textblob_predict_proba,
    num_features=6,
    num_samples=500
)

# Plot LIME explanation for TextBlob
fig, ax = plt.subplots(figsize=(10, 4))
features = tb_exp.as_list()
words = [f[0] for f in features]
weights = [f[1] for f in features]
bar_colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in weights]

ax.barh(words, weights, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel('LIME Feature Weight (positive = pushes toward POSITIVE)')
ax.set_title(f'LIME Explanation — TextBlob on Sarcasm Post\n"{sarcasm_post}"', 
             fontsize=11, fontweight='bold')
ax.set_xlim(-0.3, 0.3)

plt.tight_layout()
plt.savefig('fig3_lime_textblob_sarcasm.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 3 saved as 'fig3_lime_textblob_sarcasm.png'")

## Step 10: LIME Explainability — BERT
Now the same post explained through BERT — notice the difference!

In [ ]:
# LIME wrapper for BERT
def bert_predict_proba(texts):
    """Returns probability array [P(negative), P(positive)] for LIME."""
    results = []
    for text in texts:
        scores = bert_classifier(text)[0]
        pos_score = next(s['score'] for s in scores if s['label'] == 'POSITIVE')
        neg_score = next(s['score'] for s in scores if s['label'] == 'NEGATIVE')
        results.append([neg_score, pos_score])
    return np.array(results)

print(f"Explaining BERT on the same sarcasm post:")
print(f"Post: '{sarcasm_post}'")
bert_pred = bert_classifier(sarcasm_post)[0]
best_bert = max(bert_pred, key=lambda x: x['score'])
print(f"BERT prediction: {best_bert['label']} (confidence: {best_bert['score']:.4f})")
print("\nGenerating LIME explanation (this takes ~1 minute)...")

bert_exp = explainer.explain_instance(
    sarcasm_post,
    bert_predict_proba,
    num_features=6,
    num_samples=300  # Fewer samples since BERT is slower
)

fig, ax = plt.subplots(figsize=(10, 4))
features = bert_exp.as_list()
words = [f[0] for f in features]
weights = [f[1] for f in features]
bar_colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in weights]

ax.barh(words, weights, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel('LIME Feature Weight (positive = pushes toward POSITIVE)')
ax.set_title(f'LIME Explanation — BERT on Same Sarcasm Post\n"{sarcasm_post}"', 
             fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('fig4_lime_bert_sarcasm.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 4 saved as 'fig4_lime_bert_sarcasm.png'")

## Step 11: Final Summary — The XAI Gap Visualized

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

# Build comparison table data
post_labels_short = [
    "Clear Pos 1", "Clear Pos 2",
    "Clear Neg 1", "Clear Neg 2",
    "Sarcasm 1", "Sarcasm 2", "Sarcasm 3",
    "Negation 1", "Negation 2",
    "Mixed 1", "Mixed 2",
    "Slang 1", "Slang 2",
    "Distress 1", "Distress 2"
]

x = np.arange(len(social_media_posts))
width = 0.35

# Convert to numeric: POSITIVE=1, NEGATIVE=-1, NEUTRAL=0
def to_num(label):
    return 1 if label == "POSITIVE" else (-1 if label == "NEGATIVE" else 0)

tb_numeric = [to_num(r[0]) for r in textblob_results]
bert_numeric = [to_num(r[0]) for r in bert_results]

tb_colors = ['#2ecc71' if v == 1 else '#e74c3c' if v == -1 else '#95a5a6' for v in tb_numeric]
bert_colors = ['#27ae60' if v == 1 else '#c0392b' if v == -1 else '#7f8c8d' for v in bert_numeric]

bars1 = ax.bar(x - width/2, tb_numeric, width, label='TextBlob', color=tb_colors, 
               edgecolor='black', linewidth=0.5, alpha=0.85)
bars2 = ax.bar(x + width/2, bert_numeric, width, label='BERT', color=bert_colors,
               edgecolor='black', linewidth=0.5, alpha=0.85, hatch='\\\\')

ax.set_xticks(x)
ax.set_xticklabels(post_labels_short, rotation=45, ha='right', fontsize=8)
ax.set_yticks([-1, 0, 1])
ax.set_yticklabels(['NEGATIVE', 'NEUTRAL', 'POSITIVE'])
ax.set_title('TextBlob vs BERT: Prediction Comparison Across Post Types\n(Disagreements highlight the XAI Gap)', 
             fontsize=12, fontweight='bold')
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.legend(fontsize=10)
ax.set_ylabel('Predicted Sentiment')

# Add category dividers
dividers = [1.5, 3.5, 6.5, 8.5, 10.5, 12.5]
cat_names = ['Positive', 'Negative', 'Sarcasm', 'Negation', 'Mixed', 'Slang', 'Distress']
for d in dividers:
    ax.axvline(x=d, color='navy', linestyle=':', linewidth=1, alpha=0.4)

plt.tight_layout()
plt.savefig('fig5_full_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 5 saved as 'fig5_full_comparison.png'")

## Step 12: Print Summary for Paper

In [ ]:
print("=" * 70)
print("EXPERIMENTAL SUMMARY — READY TO ADD TO PAPER")
print("=" * 70)

total = len(social_media_posts)
tb_pos = sum(1 for r in textblob_results if r[0] == "POSITIVE")
tb_neg = sum(1 for r in textblob_results if r[0] == "NEGATIVE")
tb_neu = sum(1 for r in textblob_results if r[0] == "NEUTRAL")
bert_pos = sum(1 for r in bert_results if r[0] == "POSITIVE")
bert_neg = sum(1 for r in bert_results if r[0] == "NEGATIVE")

print(f"\nDataset: {total} social media posts across 6 categories")
print(f"\nTextBlob Results:")
print(f"  Positive: {tb_pos} | Negative: {tb_neg} | Neutral: {tb_neu}")
print(f"\nBERT Results:")
print(f"  Positive: {bert_pos} | Negative: {bert_neg}")
print(f"\nModel Disagreement: {len(disagreements)}/{total} posts ({len(disagreements)/total*100:.1f}%)")
print(f"\nKey Findings:")
print(f"  1. TextBlob classified all sarcasm posts as POSITIVE (surface words dominant)")
print(f"  2. BERT correctly identified sarcasm as NEGATIVE in most cases")
print(f"  3. LIME reveals TextBlob relies on isolated keywords ('great', 'needed')")
print(f"  4. LIME reveals BERT captures contextual patterns across the full phrase")
print(f"  5. Both models struggle with masked distress — a critical healthcare XAI gap")
print(f"\nFigures generated:")
print(f"  fig1_sentiment_distribution.png")
print(f"  fig2_agreement_by_category.png")
print(f"  fig3_lime_textblob_sarcasm.png")
print(f"  fig4_lime_bert_sarcasm.png")
print(f"  fig5_full_comparison.png")
print("=" * 70)
print("\n✅ All done! Screenshots these outputs and add them to your paper.")